# Upstream Sequence Extraction

Extracts the upstream 1,000bp region of each gene (relative to the transcription start site) from three Brassicaceae species — *Arabidopsis thaliana*, *Brassica juncea*, and *Brassica rapa* — using their FASTA and GFF annotation files. These upstream regions are the primary input for the downstream CpG island search (`preprocess.ipynb`), since CpG islands are predominantly located in promoter and 5' UTR regions.

**Input**: raw genome FASTA + GFF3 annotation files (`dataset/raw/`)
**Output**: `dataset/processed/{species}_mrna.csv` — one row per gene, containing its upstream 1,000bp sequence and coordinates

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
import pybedtools
import pandas as pd
import numpy as np
import re

In [ ]:
c_cabbage_fasta = '../dataset/raw/Brassica_rapa_ssp_pekinensis/Brapa_genome_v4.0_chrom.fasta'
c_cabbage_gff = '../dataset/raw/Brassica_rapa_ssp_pekinensis/Brapa_genome_v4.0_gene.gff3'

mustard_fasta = '../dataset/raw/Brassica_juncea/BjuT84V2.fa'
mustard_gff = '../dataset/raw/Brassica_juncea/BjuT84V2.gff'

arabidopsis_fasta = '../dataset/raw/Arabidopsis_thaliana/TAIR10_upstream_1000_20101104'
arabidopsis_gff = '../dataset/raw/Arabidopsis_thaliana/TAIR10_GFF3_genes.gff'

In [ ]:
TARGET = "mRNA"
COLUMNS =["id", "chrom", "feature", "start", "end", "strand", "seq"]

In [ ]:
def allocate_seq(row, fasta):
    """
    Extract the nucleotide sequence for a given genomic region and
    orient it according to strand direction.

    Args:
        row (pd.Series): a row containing 'chrom', 'start', 'end', 'strand'
        fasta (Bio.SeqIO.index): indexed FASTA object for sequence lookup

    Returns:
        str: uppercase nucleotide sequence. If the gene is on the '-' strand,
             the sequence is reverse-complemented so that it reads in the
             5'->3' direction of the actual transcript.
    """
    seq = Seq(fasta[row["chrom"]].seq[row["start"]:row["end"]])
    if row['strand'] == '-':
      seq = seq.reverse_complement()
    return str(seq).upper()

In [ ]:
def process_data(fasta_path, gff_path):
    """
    Extract the upstream 1,000bp region of each mRNA feature from a
    chromosome-level FASTA/GFF pair (used for B. rapa and B. juncea,
    whose FASTA files contain full chromosome sequences rather than
    pre-extracted upstream regions).

    Args:
        fasta_path (str): path to genome FASTA file
        gff_path (str): path to GFF3 annotation file

    Returns:
        pd.DataFrame: one row per mRNA feature, with columns defined in COLUMNS
    """
    #1. Get fasta, gff file from the path
    fasta = SeqIO.index(fasta_path, 'fasta')
    gff = pybedtools.BedTool(gff_path)

    #2. Extract features from gff file
    features_list = []

    for feature in gff:
        if feature[2] == TARGET:
            features_list.append({'id': feature.name,
                              'chrom': feature.chrom,
                              'feature': feature[2],
                              'start': feature.start,
                              'end': feature.end,
                              'strand': feature.strand})

    target = pd.DataFrame(features_list)

    old_start = target['start'].values
    old_end = target['end'].values

    # Redefine coordinates as the 1,000bp upstream window relative to TSS:
    # '+' strand genes -> upstream is [start-1000, start)
    # '-' strand genes -> upstream is (end, end+1000] (since transcription runs the other way)
    target['new_start'] = np.where(target['strand'] == '+', old_start - 1000, old_end)
    target['new_end'] = np.where(target['strand'] == '+', old_start, old_end + 1000)

    # Clip lower bound at 0 to prevent negative coordinates near chromosome start.
  # No upper-bound clip needed on 'end': Python/Biopython slicing safely
  # truncates out-of-range indices, so genes near the chromosome end simply
  # yield a shorter-than-1000bp upstream sequence (filtered out later by
  # the length > 200bp condition in preprocess.ipynb if too short).
    target['start'] = target['new_start'].clip(lower=0).astype(int)
    target['end'] = target['new_end'].astype(int)

    target = target.drop(columns=['new_start', 'new_end'])

    #3. Eliminate duplicates
    target = target.drop_duplicates(subset = ['chrom', 'start', 'end', 'strand'])

    #4. Extract desired sequence from fasta file
    target["seq"] = target.apply(allocate_seq, axis=1, args=(fasta,))

    #5. Return result, the dataframe
    return target[COLUMNS]

In [ ]:
#Preprocess upstream datasets
def process_upstream_data(fasta_path, gff_path):
  """
    Load pre-extracted upstream sequences (used only for Arabidopsis, whose
    FASTA file already provides one upstream 1,000bp region per gene,
    keyed by base gene ID with no isoform suffix).

    Args:
        fasta_path (str): path to pre-extracted upstream FASTA file
        gff_path (str): path to GFF3 annotation file

    Returns:
        pd.DataFrame: one row per gene, with columns defined in COLUMNS
    """
    #1. Get fasta, gff file from the path
    fasta = SeqIO.index(fasta_path, 'fasta')
    gff = pybedtools.BedTool(gff_path)

    #2. Map gff to fasta by id
    # GFF entries are per-transcript (e.g. AT1G01010.1), so we strip the
    # isoform suffix to get the base gene ID and keep only the first
    # transcript encountered per gene (matches the FASTA's one-entry-per-gene structure).
    features_map = {}
    ids = set()

    for feature in gff:
        feature_id = feature.name.split('.')[0]
        if feature[2] == TARGET and feature_id not in ids:
            features_map[feature_id] = {
                'chrom': feature.chrom,
                'feature': feature[2],
                'strand': feature.strand
            }
            ids.add(feature_id)

    #3. Join gff and fasta file
    # fasta record IDs are already base gene IDs, matching features_map keys directly.
    features_list = []

    for i in list(fasta):
        if i in features_map:
            feature = features_map[i]
            # FASTA header encodes coordinates as e.g. "1-1000"; fall back to
            # a default window if parsing fails (should not normally occur).
            coords = re.findall(r'(\d+)-(\d+)', fasta[i].description)
            if coords:
              start = int(coords[0][0])
              end = int(coords[0][1])
            else:
              print(f"Warning: could not parse coordinates for {i}, using default (0, 1000)")
              start, end = 0, 1000

            features_list.append({
                "id": i,
                "chrom": feature["chrom"],
                "feature": feature["feature"],
                "start": start,
                "end": end,
                "strand": feature["strand"],
                "seq": str(fasta[i].seq).upper()
                })

    target = pd.DataFrame(features_list)

    #4. Return result
    return target[COLUMNS]

In [ ]:
def save_csv():
    """
    Run upstream sequence extraction for all three species and save each
    result to dataset/processed/. Arabidopsis uses process_upstream_data
    (pre-extracted upstream FASTA, one entry per gene); B. juncea and
    B. rapa use process_data (chromosome-level FASTA, upstream window
    computed manually from GFF coordinates).
    """
    print('Starting preprocessing.')
    arabidopsis = process_upstream_data(arabidopsis_fasta, arabidopsis_gff)
    print('Ended preprocessing arabidopsis file.')
    mustard = process_data(mustard_fasta, mustard_gff)
    print('Ended preprocessing mustard file.')
    c_cabbage = process_data(c_cabbage_fasta, c_cabbage_gff)
    print('Ended preprocessing cabbage file.')

    arabidopsis.to_csv("../dataset/processed/arabidopsis_mrna.csv", index=False)
    mustard.to_csv("../dataset/processed/mustard_mrna.csv", index=False)
    c_cabbage.to_csv("../dataset/processed/c_cabbage_mrna.csv", index=False)
    print("Saved preprocessed datasets at '../dataset/processed'.")

In [ ]:
save_csv()

Starting preprocessing.
Ended preprocessing arabidopsis file.
Ended preprocessing mustard file.
Ended preprocessing cabbage file.
Saved preprocessed datasets at '../dataset/processed'.
